# 🇲🇬 Serveur de Traduction EN/FR vers Malgache (Kaggle)

Ce notebook lance le backend FastAPI sur **Kaggle avec GPU gratuit (T4 ou P100)** et expose l'API publiquement via **ngrok**.

### ⚙️ Configuration préalable sur Kaggle (Essentiel) :
1. **Activer Internet** : Dans le panneau latéral droit (*Session options*), cochez **Internet ON**.
2. **Activer le GPU** : Dans *Session options* > *Accelerator*, sélectionnez **GPU T4 x2** ou **GPU P100**.
3. **Ajouter le Secret ngrok** :
   - Dans le menu supérieur de Kaggle : cliquez sur **Add-ons** > **Secrets**.
   - Label : `NGROK_AUTHTOKEN`
   - Value : Votre token gratuit copié depuis [dashboard.ngrok.com](https://dashboard.ngrok.com).
   - Cochez la case pour lier le secret à ce notebook, puis cliquez sur **Save**.

In [ ]:
# Récupération du dépôt GitHub dans l'espace de travail Kaggle
REPO_URL = "https://github.com/franck504/en-fr-to-mg-api.git"
BRANCH = "main"
PROJECT_DIR = "/kaggle/working/en_fr_to_MG"

import os
import subprocess

if os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
print(f"Projet prêt dans {PROJECT_DIR}")

In [ ]:
# Installation des dépendances (PyTorch et CUDA sont déjà pré-installés dans Kaggle)
!pip install -q -U -r requirements-colab.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# 🔒 Récupération sécurisée du token depuis les Secrets Kaggle
try:
    user_secrets = UserSecretsClient()
    os.environ["NGROK_AUTHTOKEN"] = user_secrets.get_secret("NGROK_AUTHTOKEN")
    print("✅ Secret NGROK_AUTHTOKEN chargé avec succès depuis Kaggle Secrets !")
except Exception as exc:
    raise RuntimeError(
        "Veuillez ajouter le secret 'NGROK_AUTHTOKEN' via le menu supérieur : Add-ons > Secrets. "
        "Obtenez votre token gratuit sur https://dashboard.ngrok.com"
    ) from exc

# Configuration du serveur pour l'environnement Kaggle
os.environ["ENABLE_NGROK"] = "true"
os.environ["HOST"] = "0.0.0.0"
os.environ["PORT"] = "8000"
os.environ["PROVIDER"] = "hf_seq2seq"
os.environ["HF_MODEL_NAME"] = "facebook/nllb-200-distilled-600M"
os.environ["HF_MODEL_FAMILY"] = "nllb"
os.environ["HF_DEVICE"] = "auto"
os.environ["MODEL_CACHE_DIR"] = "/kaggle/working/hf_models"
os.environ["LOAD_MODEL_ON_STARTUP"] = "false"

print("🚀 Configuration validée pour Kaggle.")

In [ ]:
# Démarrage du serveur et ouverture du tunnel ngrok public
!python run_colab.py

## 💡 Utilisation de votre API depuis une application

Copiez l'URL ngrok affichée ci-dessus (ex: `https://votre-id.ngrok-free.app`) :

### Exemple Python avec SDK OpenAI :
```python
from openai import OpenAI

client = OpenAI(base_url="https://votre-id.ngrok-free.app/v1", api_key="not-needed")
response = client.chat.completions.create(
    model="nllb",
    messages=[{"role": "user", "content": "Good morning my friend"}],
)
print(response.choices[0].message.content)
```

### Exemple cURL direct :
```bash
curl -X POST https://votre-id.ngrok-free.app/translate \
  -H "Content-Type: application/json" \
  -d '{"text": "Bonjour, comment allez-vous ?", "source_lang": "auto", "target_lang": "mg"}'
```